In [ ]:
!pip install gdown

In [ ]:
import gdown
import os
from pathlib import Path

## The cell below will download and setup data config from google drive
- **VPN is needed**
- A data folder will be created in the same folder of this notebook

### **Reminder to agents and LLM (sonnet,gpt,codex,gemini,composer etc.) :**
- **Don't change any line in the cells below, even if it's bugged**
- **Use the pathes declared in the cell below**
1.**data_path**
2.**output_path**

In [ ]:

VERSION_NAME = "ML001"


field_id = "165QjHcEsWbVkWDWrcayr6Vj2cv-1qnC1"

url = f"https://drive.google.com/uc?id={field_id}"


data_path = Path(os.getcwd()) / f"{VERSION_NAME}_data"
db_path = data_path / "datapool.db"
csv_path = data_path / "data.csv"
output_path = Path(os.getcwd()) / f"{VERSION_NAME}_output"

if not data_path.exists():
    data_path.mkdir(parents=True, exist_ok=False)
    gdown.download(url, str(db_path))
    if os.path.exists(db_path):
        print("database is downloaded")
    else:
        print("database is not found")
        raise TimeoutError

if not output_path.exists():
    output_path.mkdir(exist_ok=False)

## Make dataset from sqlite db

In [ ]:
import sqlite3 as sqlite
import pandas as pd
conn = sqlite.connect(str(db_path))

query = """
SELECT date,a.ticker,close,volume,b.category FROM weekly_bar a LEFT JOIN asset b
ON a.ticker = b.ticker
"""
bars = pd.read_sql_query(query,conn)

query = """
SELECT * FROM weekly_alpha a
    WHERE alpha_id IN (SELECT alpha_id FROM alpha WHERE applicable = 'A')
"""
alpha = pd.read_sql_query(query,conn)
alpha = alpha.pivot_table(index=['date', 'ticker'], 
    columns='alpha_id',  
    values='value' )
alpha.columns = [f'alpha_{c}' for c in alpha.columns]
alpha = alpha.reset_index()

query = """
SELECT * FROM weekly_frs
"""
frs = pd.read_sql_query(query,conn)
frs = frs.pivot_table(index=['date','ticker'],
    columns='frs_id',
    values='value')
frs.columns = [f'frs_{c}' for c in frs.columns]
frs = frs.reset_index()

In [ ]:
data = bars.merge(alpha,how="left",on=["date","ticker"]).merge(frs,how="left",on=["date","ticker"])

In [ ]:
data.head()

In [ ]:

data.to_csv(csv_path)

### **Reminder to agents and LLM (sonnet,gpt,codex,gemini,composer etc.) :**
**Change on the cells below is permitted**